In [ ]:
# ── Cell 1 · Environment Setup ───────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
# Paste into ONE Google Colab cell (Runtime > Change runtime type > CPU) and run.
# Loads YOUR trained models from Google Drive (.../cnn_lstm/<bundle>/final_model.keras)
# and measures how long the ensemble takes to score ONE 60-second window.
import os, glob, time, numpy as np
from google.colab import drive
drive.mount('/content/drive')
from tensorflow import keras

SENSOR_MODELS = ['chest_acc', 'chest_eda', 'chest_emg', 'chest_heart', 'chest_statistical',
                 'wrist_acc', 'wrist_eda', 'wrist_heart', 'wrist_statistical']
FUSED_MODELS  = ['chest_all', 'wrist_all']

def find(bundle):
    hits = glob.glob(f'/content/drive/MyDrive/**/cnn_lstm/{bundle}/final_model.keras', recursive=True)
    if not hits:
        raise FileNotFoundError(f'Could not find cnn_lstm/{bundle}/final_model.keras in your Drive')
    return hits[0]

models = {}
for b in SENSOR_MODELS + FUSED_MODELS:
    p = find(b); models[b] = keras.models.load_model(p, compile=False)
    print(f'Loaded {b:18s} input={models[b].input_shape}  <- {p}')

def dummy_input(m):  # random features with the right shape; speed does not depend on values
    return np.random.randn(1, *m.input_shape[1:]).astype('float32')

def time_ensemble(names, n=500):
    xs = {k: dummy_input(models[k]) for k in names}
    for _ in range(20):                                      # warm-up
        for k in names: models[k](xs[k], training=False)
    t = []
    for _ in range(n):                                       # score n windows, one at a time
        t0 = time.perf_counter()
        prob = np.mean([float(models[k](xs[k], training=False)[0, 0]) for k in names])  # probability averaging
        t.append((time.perf_counter() - t0) * 1000)
    return np.median(t), np.percentile(t, 95)

def params(names): return sum(models[k].count_params() for k in names)

print('\n===== COPY THESE INTO THE PAPER =====')
for label, names in [('Deployed system (9 sensor models averaged)', SENSOR_MODELS),
                     ('Reported fused model (chest_all + wrist_all)', FUSED_MODELS)]:
    med, p95 = time_ensemble(names)
    print(f'{label}\n   median {med:.0f} ms per window | 95th percentile {p95:.0f} ms | '
          f'{params(names):,} parameters (~{params(names)*4/1e6:.1f} MB)')
print('CPU:', os.popen("lscpu | grep 'Model name'").read().split(':')[-1].strip())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded chest_acc          input=(None, 1, 23)  <- /content/drive/MyDrive/dl_models/cnn_lstm/chest_acc/final_model.keras
Loaded chest_eda          input=(None, 1, 8)  <- /content/drive/MyDrive/dl_models/cnn_lstm/chest_eda/final_model.keras
Loaded chest_emg          input=(None, 1, 12)  <- /content/drive/MyDrive/dl_models/cnn_lstm/chest_emg/final_model.keras
Loaded chest_heart        input=(None, 1, 10)  <- /content/drive/MyDrive/dl_models/cnn_lstm/chest_heart/final_model.keras
Loaded chest_statistical  input=(None, 1, 117)  <- /content/drive/MyDrive/dl_models/cnn_lstm/chest_statistical/final_model.keras
Loaded wrist_acc          input=(None, 1, 23)  <- /content/drive/MyDrive/dl_models/cnn_lstm/wrist_acc/final_model.keras
Loaded wrist_eda          input=(None, 1, 8)  <- /content/drive/MyDrive/dl_models/cnn_lstm/wrist_eda/final_model.keras
Loaded wrist_heart    